# Three Tests

This notebook implements three non-trivial tests to validate the period recovery pipeline:

1. **Analytic Sine Recovery**: Verification of period recovery for a perfect sine wave.

2. **Dynamic Beta Estimation**: Testing if the code can recover a known phase slope from noisy synthetic data.

3. **Real-World Benchmark**: Validating the pipeline against the rotation period of asteroid 4 Vesta (*P* = 5.342 h).

**Import Statements**

In [5]:
import sys
import importlib
import os
import numpy as np
import matplotlib.pyplot as plt
sys.path.append(os.path.abspath(".."))

import pipeline.io as io
import pipeline.data_handler as dh
import pipeline.fourier_engine as fe
import pipeline.harmonic_logic as hl
import pipeline.output_gen as dh

import matplotlib.pyplot as plt

# First Test

1. **Analytic Sine Recovery**: Verification of period recovery for a perfect sine wave.


User defined parameters:

In [39]:
# increasing the following two will greatly increase computation time
# below the sample rate and duration are equivalent to 20 nights of data with 1000 data points each
# this is pretty realistic compared to the data we have used 

just_sine_wave = True # Keep this on true just to run the first test, the others will bump up the execution time

sample_rate = 1000  # Number of data points per second
duration = 20 # Length of data in seconds
periods = np.array([10.0 ,28.0, 30.0, 35.0, 40.0]) # Test periods in seconds

### Fourier coefficients, the first is a perfect sine wave with period of 10 seconds###
C_coeffs = np.array([0.0, 10.2, 20.1, -35.3, 40.8])
A1_coeffs = np.array([1.0, 20.0, 0.0, 5.0, 10.0])
A2_coeffs = np.array([0.0, 10.2, 15.4, 16.3, 19.6])
B1_coeffs = np.array([0.0, -4.4, 5.6, 8.9, 10.2])
B2_coeffs = np.array([0.0, -40.0, 3.1, 6.2, 7.2])

# First wave is perfect sine wave, rest are double peaked 'asteroid light curves'
is_double_peaked = np.array([False, True, True, True, True]) 


frequencies = periods ** -1 #Convert frequnecies to periods
print(f'Frequencies array: {frequencies}')

Frequencies array: [0.1        0.03571429 0.03333333 0.02857143 0.025     ]


The actual test following the same pipeline as the asteroid data after calibration. Not the period is given in hours, this is due to the convention of the data passed in as being in JD, for this test, time units are arbitrary, and the period will be off by a factor of 24 purely due to a unit mismatch. For best comparison, we stick with frequnecy.

In [ ]:

for i, (C, A_1, A_2, B_1, B_2, freq, is_doub_true) in enumerate(zip(C_coeffs, A1_coeffs, A2_coeffs, B1_coeffs, B2_coeffs, frequencies, is_double_peaked)):

    if just_sine_wave and i > 0:
        break
    
    print(f'\n Starting test {i+1}. \n')

    t = np.linspace(0, duration, sample_rate * duration, endpoint=False)
    data = C + A_1 * np.sin(2 * np.pi * freq * t) + B_1 * np.cos(2*np.pi * freq*t) + A_2 * np.sin(4 * np.pi * freq * t) + B_2 * np.cos(4*np.pi*freq*t)

    # Following the method on the already calibrated data in the real pipeline, we first test
    # If lomb scargle recovers the correct frequnecy, likely double the true frequency due to the double peaked nature
    # We use a frequency min and max that encompases the data corrected
    freqs, powers, fpeak = fe.lomb_scargle(t, data, fmin=0.01, fmax=1) 
    print(f'Lomb scargle peak frequnecy: {fpeak}')

    # Then we test our fourier refinement using the peak lomb scargle frequencies
    best_frot, period_hours, best_coeffs, is_double = hl.harmonic_logic(t, data, freqs, powers)

    print(f'Found coefficients \n A1: {best_coeffs[1]} \n B1: {best_coeffs[2]}'
          f'\n A2: {best_coeffs[3]} \n B2: {best_coeffs[4]}')
    
    # compute the errors

    true_coeffs = np.array([C, A_1, B_1, A_2, B_2])
    coeff_errors = np.abs(best_coeffs - true_coeffs)
    coeff_rel_errors = coeff_errors / np.abs(true_coeffs + 1e-10)

    freq_error = np.abs(best_frot - freq)
    freq_rel_error = freq_error / freq

    print(f"Freq error:  abs={freq_error:.6f} c/d  rel={freq_rel_error*100:.4f}%")
    print(f"Coeff errors (abs): C={coeff_errors[0]:.4f}  A1={coeff_errors[1]:.4f}  B1={coeff_errors[2]:.4f}  A2={coeff_errors[3]:.4f}  B2={coeff_errors[4]:.4f}")
    
    if is_double == is_doub_true:
        print(f'Correctly characterized double peaked as: {is_double}.')
    else:
        print(f'Incorrectly characterized double peaked as: {is_double}.')
    
    print(f'\n Finished test {i+1}. \n')



 Starting test 1. 

Lomb scargle peak frequnecy: 0.10000450022501126
Found 2 clusters, refining each...
  f_ls=0.1000 c/d | single: frot=0.1000 (P=240.001 hr) chi2=0.000000 R2=0.0000 | double: frot=0.0500 (P=480.002 hr) chi2=0.000000 R2=149850.9828
  f_ls=0.0100 c/d | single: frot=0.0110 (P=2181.818 hr) chi2=0.346567 R2=0.5431 | double: frot=0.0055 (P=4363.636 hr) chi2=0.353698 R2=0.5099

Best rotation frequency: 0.099999 c/d
Best period: 240.0012 hours
Chi2 single: 0.000000
Chi2 double: 0.000000
Harmonic amplitude ratio: 0.0000
Double peaked: False
Found coefficients 
 A1: 0.9999974968345162 
 B1: 3.144412356243264e-05
 A2: -6.673389088946287e-06 
 B2: -3.564467976584046e-09
Freq error:  abs=0.000001 c/d  rel=0.0005%
Coeff errors (abs): C=0.0000  A1=0.0000  B1=0.0000  A2=0.0000  B2=0.0000
Correctly characterized double peaked as: False.

 Finished test 1. 


 Starting test 2. 



# First Test

1. **Analytic Sine Recovery**: Verification of period recovery for a perfect sine wave.


User defined parameters: